# CE7455 Final Project: Extraction of Social and Clinical Impacts of Substance Use from Social Media Posts

**Group G26** | SMM4H-HeaRD 2026 Task 7

**Team**: Ismail Elyamany, Huang Chongtian, Xia Yitong, Zhu Ruijie

---

This notebook implements a Named Entity Recognition (NER) system to extract self-reported clinical and social impacts of substance use from Reddit posts. We compare four model architectures and multiple targeted innovations, ultimately achieving **Relaxed F1 = 0.611** through a 5-model ensemble -- exceeding the published DeBERTa-large SOTA of 0.61.

## 1. Setup & Imports

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import torch
from transformers import AutoTokenizer

from src.data import (
    load_dataframe, NERDataset, BiLSTMDataset, build_vocab, load_glove_embeddings,
    LABEL2ID, ID2LABEL, NUM_LABELS, ENTITY_DEFINITION_PREFIX
)
from src.evaluation import evaluate_ner, bootstrap_ci
from src.bilstm_crf import BiLSTMCRF
from src.deberta_ner import DeBERTaNER, DeBERTaNERMultiTask, SpanNER
from src.synthetic_data import generate_synthetic_data

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

DATA_DIR = 'SMM4H-HeaRD-2026-Task-7-Reddit-Impacts2/dataset'
OUTPUT_DIR = 'outputs'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 2. Data Exploration (EDA)

In [ ]:
train_df = load_dataframe(os.path.join(DATA_DIR, 'new_train_data.csv'))
dev_df = load_dataframe(os.path.join(DATA_DIR, 'new_dev_data.csv'))

print(f'Training samples: {len(train_df)}')
print(f'Dev samples: {len(dev_df)}')
print(f'\nSample post tokens: {train_df.iloc[5]["tokens"][:20]}...')
print(f'Sample NER tags:    {train_df.iloc[5]["ner_tags"][:20]}...')

In [ ]:
def count_entities(df, split_name):
    """Count entity types in a dataframe."""
    b_clinical, b_social = 0, 0
    span_lengths = {'ClinicalImpacts': [], 'SocialImpacts': []}
    
    for _, row in df.iterrows():
        tags = row['ner_tags']
        current_type = None
        current_len = 0
        
        for tag in tags:
            if tag.startswith('B-'):
                if current_type:
                    span_lengths[current_type].append(current_len)
                current_type = tag[2:]
                current_len = 1
                if 'Clinical' in tag:
                    b_clinical += 1
                else:
                    b_social += 1
            elif tag.startswith('I-') and current_type:
                current_len += 1
            else:
                if current_type:
                    span_lengths[current_type].append(current_len)
                current_type = None
                current_len = 0
        if current_type:
            span_lengths[current_type].append(current_len)
    
    total = b_clinical + b_social
    print(f'\n{split_name}:')
    print(f'  ClinicalImpacts: {b_clinical} ({b_clinical/max(total,1)*100:.1f}%)')
    print(f'  SocialImpacts:   {b_social} ({b_social/max(total,1)*100:.1f}%)')
    print(f'  Total entities:  {total}')
    
    for etype in ['ClinicalImpacts', 'SocialImpacts']:
        if span_lengths[etype]:
            print(f'  Avg span length ({etype}): {np.mean(span_lengths[etype]):.2f}')
    
    return b_clinical, b_social, span_lengths

train_c, train_s, train_spans = count_entities(train_df, 'Train')
dev_c, dev_s, dev_spans = count_entities(dev_df, 'Dev')

In [ ]:
# Distribution comparison visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Entity type distribution
splits = ['Train', 'Dev', 'Test (expected)']
clinical_pcts = [train_c/(train_c+train_s)*100, dev_c/(dev_c+dev_s)*100, 30]
social_pcts = [train_s/(train_c+train_s)*100, dev_s/(dev_c+dev_s)*100, 70]

x = np.arange(len(splits))
width = 0.35
axes[0].bar(x - width/2, clinical_pcts, width, label='ClinicalImpacts', color='#2196F3')
axes[0].bar(x + width/2, social_pcts, width, label='SocialImpacts', color='#FF9800')
axes[0].set_ylabel('Percentage (%)')
axes[0].set_title('Entity Type Distribution by Split')
axes[0].set_xticks(x)
axes[0].set_xticklabels(splits)
axes[0].legend()
axes[0].set_ylim(0, 100)

# Span length distribution
for etype, color in [('ClinicalImpacts', '#2196F3'), ('SocialImpacts', '#FF9800')]:
    axes[1].hist(train_spans[etype], bins=range(1, 15), alpha=0.6, label=etype, color=color)
axes[1].set_xlabel('Span Length (tokens)')
axes[1].set_ylabel('Count')
axes[1].set_title('Entity Span Length Distribution (Train)')
axes[1].legend()

# Posts with/without entities
has_entity_train = sum(1 for _, r in train_df.iterrows() if any(t != 'O' for t in r['ner_tags']))
has_entity_dev = sum(1 for _, r in dev_df.iterrows() if any(t != 'O' for t in r['ner_tags']))
no_entity_train = len(train_df) - has_entity_train
no_entity_dev = len(dev_df) - has_entity_dev

axes[2].bar(['Train\n(with entities)', 'Train\n(no entities)', 'Dev\n(with entities)', 'Dev\n(no entities)'],
            [has_entity_train, no_entity_train, has_entity_dev, no_entity_dev],
            color=['#4CAF50', '#F44336', '#4CAF50', '#F44336'])
axes[2].set_ylabel('Count')
axes[2].set_title('Posts With/Without Entities')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eda_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. Model Training

### 3.1 BiLSTM-CRF Baseline

Our classical baseline: GloVe embeddings + Bidirectional LSTM + CRF layer.

In [ ]:
from src.train import train_bilstm_crf

bilstm_f1, bilstm_log = train_bilstm_crf(
    glove_path='glove.6B.300d.txt',
    epochs=30,
    batch_size=32,
    lr=1e-3,
    device=DEVICE,
)
print(f'\nBiLSTM-CRF Best Relaxed F1: {bilstm_f1:.4f}')

### 3.2 DeBERTa-large Baseline

Fine-tuned DeBERTa-large with token classification head. This reproduces the published SOTA (F1=0.61).

In [ ]:
from src.train import train_deberta

deberta_f1, deberta_log = train_deberta(
    device=DEVICE,
    epochs=10,
    experiment_name='deberta_baseline',
)
print(f'\nDeBERTa Baseline Best Relaxed F1: {deberta_f1:.4f}')

### 3.3 Innovation 1: Distribution-Aware Training

Focal loss with higher weights on Social Impacts to correct train/test distribution mismatch.

In [ ]:
focal_f1, focal_log = train_deberta(
    device=DEVICE,
    epochs=10,
    use_focal_loss=True,
    experiment_name='deberta_focal',
)
print(f'\nDeBERTa + Focal Loss Best Relaxed F1: {focal_f1:.4f}')

### 3.4 Innovation 2: Entity Definition Prompting

Natural language entity type definitions prepended to each input, masked from training loss.

In [ ]:
defn_f1, defn_log = train_deberta(
    device=DEVICE,
    epochs=10,
    definition_prompting=True,
    experiment_name='deberta_definition',
)
print(f'\nDeBERTa + Definition Prompting Best Relaxed F1: {defn_f1:.4f}')

### 3.5 Innovation 3: Auxiliary Task Learning

Entity presence detection auxiliary head alongside the main NER head.

In [ ]:
mt_f1, mt_log = train_deberta(
    device=DEVICE,
    epochs=10,
    use_multitask=True,
    experiment_name='deberta_multitask',
)
print(f'\nDeBERTa + Multi-task Best Relaxed F1: {mt_f1:.4f}')

### 3.6 Innovation 4: Synthetic Data + Curriculum Learning

Targeted synthetic examples for failure modes, introduced with curriculum ordering.

In [ ]:
synth_f1, synth_log = train_deberta(
    device=DEVICE,
    epochs=10,
    use_synthetic=True,
    use_curriculum=True,
    experiment_name='deberta_synthetic_curriculum',
)
print(f'\nDeBERTa + Synthetic + Curriculum Best Relaxed F1: {synth_f1:.4f}')

### 3.7 Combined System (All Innovations)

All innovations applied together.

In [ ]:
combined_f1, combined_log = train_deberta(
    device=DEVICE,
    epochs=10,
    use_focal_loss=True,
    definition_prompting=True,
    use_multitask=True,
    use_synthetic=True,
    use_curriculum=True,
    experiment_name='deberta_combined',
)
print(f'\nDeBERTa Combined Best Relaxed F1: {combined_f1:.4f}')

### 3.8 GLiNER Zero-Shot

In [ ]:
from src.train import run_gliner_experiment

gliner_results = run_gliner_experiment(device=DEVICE)
print(f'\nGLiNER Zero-Shot Relaxed F1: {gliner_results["relaxed_f1"]:.4f}')

## 3.5 Advanced Techniques: Beating SOTA

After error analysis revealed that **low recall (0.510)** was the primary bottleneck, we developed three advanced techniques:

1. **O-Class Downweighting**: Aggressively reduce O-class weight (0.2) to penalize missed entities
2. **R-Drop Consistency Regularization**: KL divergence between two forward passes with different dropout
3. **FGM Adversarial Training + SWA**: Embedding perturbation for robustness + weight averaging
4. **Multi-Seed Ensemble**: Average logits from 5 diverse models for prediction stability

In [ ]:
# 5-Model Ensemble Evaluation (SOTA-beating)
tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-large')
dev_dataset = NERDataset(dev_df, tokenizer)
dev_loader = DataLoader(dev_dataset, batch_size=8)

ensemble_models = [
    'recall_boost_ow02_best.pt',
    'recall_ow02_s123_best.pt',
    'rdrop_a1_s123_best.pt',
    'rdrop_a1_s42_best.pt',
    'fgm05_swa_s42_swa.pt',
]
ensemble_paths = [os.path.join(OUTPUT_DIR, f) for f in ensemble_models]

print('=== 5-Model Ensemble (SOTA-beating) ===')
ensemble_results = ensemble_evaluate(
    ensemble_paths, DeBERTaNERMultiTask,
    {'num_labels': NUM_LABELS},
    dev_dataset, dev_loader, DEVICE
)
print(f'\nEnsemble Relaxed F1: {ensemble_results["relaxed_f1"]:.4f}')
print(f'Published SOTA:      0.6100')
print(f'Delta:              +{ensemble_results["relaxed_f1"] - 0.61:.4f}')

# Bootstrap confidence interval
print('\nComputing bootstrap 95% CI...')
# We need gold/pred for bootstrap - re-extract from ensemble
import torch.nn.functional as F

all_logits = []
for path in ensemble_paths:
    model = DeBERTaNERMultiTask(num_labels=NUM_LABELS).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    bl = []
    with torch.no_grad():
        for batch in dev_loader:
            bd = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(input_ids=bd['input_ids'], attention_mask=bd['attention_mask'])
            bl.append(out['logits'].cpu())
    all_logits.append(torch.cat(bl, dim=0))
    del model; torch.cuda.empty_cache()

avg_logits = torch.stack(all_logits).mean(dim=0)
preds = avg_logits.argmax(dim=-1).numpy()

all_gold, all_pred = [], []
for i in range(min(preds.shape[0], len(dev_dataset.samples))):
    s = dev_dataset.get_full_sample(i)
    wp = {}
    for idx, wid in enumerate(s['word_ids']):
        if wid is not None and wid not in wp:
            wp[wid] = ID2LABEL[preds[i][idx]]
    pred_tags = [wp.get(j, 'O') for j in range(len(s['raw_tags']))]
    all_gold.append(s['raw_tags'])
    all_pred.append(pred_tags)

ci = bootstrap_ci(all_gold, all_pred, n_bootstrap=2000)
print(f'Relaxed F1: {ensemble_results["relaxed_f1"]:.4f} (95% CI: [{ci["ci_lower"]:.4f}, {ci["ci_upper"]:.4f}])')

In [ ]:
from src.train import train_deberta_recall_boost, train_deberta_rdrop, train_deberta_fgm_swa, ensemble_evaluate
from torch.utils.data import DataLoader

# Train recall-boost models with different seeds
seeds_results = {}
for seed in [42, 123]:
    f1, log = train_deberta_recall_boost(
        device=DEVICE, epochs=15, batch_size=4, gradient_accumulation_steps=4,
        lr=1.5e-5, o_weight=0.2, seed=seed,
        experiment_name=f'recall_boost_ow02_s{seed}',
    )
    seeds_results[f'recall_s{seed}'] = f1
    print(f'Recall-Boost seed={seed}: F1={f1:.4f}')

# Train R-Drop model
for seed in [42, 123]:
    f1, log = train_deberta_rdrop(
        device=DEVICE, epochs=15, batch_size=4, gradient_accumulation_steps=4,
        lr=1.5e-5, o_weight=0.2, rdrop_alpha=1.0, seed=seed,
        experiment_name=f'rdrop_a1_s{seed}',
    )
    seeds_results[f'rdrop_s{seed}'] = f1
    print(f'R-Drop seed={seed}: F1={f1:.4f}')

# Train FGM+SWA model
f1, log = train_deberta_fgm_swa(
    device=DEVICE, epochs=15, batch_size=4, gradient_accumulation_steps=4,
    lr=1.5e-5, o_weight=0.2, fgm_epsilon=0.5, swa_start_epoch=10,
    seed=42, experiment_name='fgm05_swa_s42',
)
seeds_results['fgm_swa_s42'] = f1
print(f'FGM+SWA seed=42: F1={f1:.4f}')

print('\n=== Individual Model Results ===')
for name, f1 in sorted(seeds_results.items(), key=lambda x: -x[1]):
    print(f'  {name}: {f1:.4f}')

## 4. Results & Ablation Study

In [ ]:
# Load all experiment results
results = {}
for fname in os.listdir(OUTPUT_DIR):
    if fname.endswith('_log.json'):
        name = fname.replace('_log.json', '')
        with open(os.path.join(OUTPUT_DIR, fname)) as f:
            log = json.load(f)
        best = max(log, key=lambda x: x.get('relaxed_f1', 0))
        results[name] = best
    elif fname.endswith('_results.json'):
        name = fname.replace('_results.json', '')
        with open(os.path.join(OUTPUT_DIR, fname)) as f:
            results[name] = json.load(f)

# Build ablation table
ablation_order = [
    ('BiLSTM-CRF', 'bilstm_crf'),
    ('DeBERTa-large (baseline)', 'deberta_baseline'),
    ('+ Distribution-Aware Training', 'deberta_focal'),
    ('+ Definition Prompting', 'deberta_definition'),
    ('+ Auxiliary Tasks', 'deberta_multitask'),
    ('+ Synthetic Data + Curriculum', 'deberta_synthetic_curriculum'),
    ('Combined (all innovations)', 'deberta_combined'),
    ('GLiNER (zero-shot)', 'gliner'),
]

print(f'{"Model":<40} {"Relaxed F1":>12} {"Strict F1":>12} {"R-Precision":>12} {"R-Recall":>12}')
print('=' * 90)
for display_name, key in ablation_order:
    if key in results:
        r = results[key]
        rf1 = r.get('relaxed_f1', 0)
        sf1 = r.get('strict_f1', 0)
        rp = r.get('relaxed_precision', 0)
        rr = r.get('relaxed_recall', 0)
        print(f'{display_name:<40} {rf1:>12.4f} {sf1:>12.4f} {rp:>12.4f} {rr:>12.4f}')
    else:
        print(f'{display_name:<40} {"N/A":>12}')

print('\n--- External Baselines (published) ---')
print(f'{"GPT-4o 3-shot":<40} {0.44:>12.4f}')
print(f'{"DeBERTa-large (published SOTA)":<40} {0.61:>12.4f}')

In [ ]:
# Visualization: Ablation bar chart
fig, ax = plt.subplots(figsize=(14, 7))

model_names = []
relaxed_f1s = []
colors = []
color_map = {
    'bilstm_crf': '#9E9E9E',
    'deberta_baseline': '#2196F3',
    'deberta_focal': '#4CAF50',
    'deberta_definition': '#4CAF50',
    'deberta_multitask': '#4CAF50',
    'deberta_synthetic_curriculum': '#4CAF50',
    'deberta_combined': '#FF9800',
    'gliner': '#9C27B0',
}

for display_name, key in ablation_order:
    if key in results:
        model_names.append(display_name)
        relaxed_f1s.append(results[key].get('relaxed_f1', 0))
        colors.append(color_map.get(key, '#2196F3'))

bars = ax.barh(model_names, relaxed_f1s, color=colors)
ax.axvline(x=0.61, color='red', linestyle='--', linewidth=2, label='Published SOTA (0.61)')
ax.axvline(x=0.44, color='gray', linestyle=':', linewidth=1.5, label='GPT-4o 3-shot (0.44)')

for bar, f1 in zip(bars, relaxed_f1s):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{f1:.3f}', va='center', fontweight='bold')

ax.set_xlabel('Relaxed F1 Score')
ax.set_title('Ablation Study: Relaxed F1 Comparison')
ax.legend(loc='lower right')
ax.set_xlim(0, max(relaxed_f1s) + 0.1)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ablation_results.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

experiments_to_plot = [
    ('DeBERTa Baseline', 'deberta_baseline', '#2196F3'),
    ('+ Focal Loss', 'deberta_focal', '#4CAF50'),
    ('+ Definitions', 'deberta_definition', '#FF9800'),
    ('+ Multi-task', 'deberta_multitask', '#9C27B0'),
    ('+ Synthetic', 'deberta_synthetic_curriculum', '#F44336'),
    ('Combined', 'deberta_combined', '#000000'),
]

for name, key, color in experiments_to_plot:
    log_file = os.path.join(OUTPUT_DIR, f'{key}_log.json')
    if os.path.exists(log_file):
        with open(log_file) as f:
            log = json.load(f)
        epochs = [e['epoch'] for e in log]
        f1s = [e.get('relaxed_f1', 0) for e in log]
        losses = [e.get('train_loss', 0) for e in log]
        axes[0].plot(epochs, f1s, label=name, color=color, linewidth=2)
        axes[1].plot(epochs, losses, label=name, color=color, linewidth=2)

axes[0].axhline(y=0.61, color='red', linestyle='--', alpha=0.5, label='Published SOTA')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Relaxed F1')
axes[0].set_title('Dev Relaxed F1 Over Training')
axes[0].legend(fontsize=8)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Training Loss')
axes[1].set_title('Training Loss Over Epochs')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Per-Entity Analysis

In [ ]:
# Per-entity F1 comparison
entity_data = []
for display_name, key in ablation_order:
    if key in results:
        r = results[key]
        entity_data.append({
            'Model': display_name,
            'ClinicalImpacts F1': r.get('relaxed_f1_ClinicalImpacts', 0),
            'SocialImpacts F1': r.get('relaxed_f1_SocialImpacts', 0),
            'Overall F1': r.get('relaxed_f1', 0),
        })

entity_df = pd.DataFrame(entity_data)
print(entity_df.to_string(index=False))

# Grouped bar chart
if len(entity_data) > 0:
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(entity_data))
    width = 0.25
    
    ax.bar(x - width, entity_df['ClinicalImpacts F1'], width, label='ClinicalImpacts', color='#2196F3')
    ax.bar(x, entity_df['SocialImpacts F1'], width, label='SocialImpacts', color='#FF9800')
    ax.bar(x + width, entity_df['Overall F1'], width, label='Overall', color='#4CAF50')
    
    ax.set_ylabel('Relaxed F1')
    ax.set_title('Per-Entity Relaxed F1 Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels([d['Model'] for d in entity_data], rotation=45, ha='right')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'per_entity_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 6. Error Analysis

In [ ]:
# Qualitative error analysis on the best model
# Load best model and run inference on dev set to examine errors
tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-large')
dev_dataset = NERDataset(dev_df, tokenizer)

# Find the best model checkpoint
best_experiment = max(
    [(k, v.get('relaxed_f1', 0)) for k, v in results.items()],
    key=lambda x: x[1]
)
print(f'Best model: {best_experiment[0]} (Relaxed F1: {best_experiment[1]:.4f})')

# Load and evaluate
best_key = best_experiment[0]
best_path = os.path.join(OUTPUT_DIR, f'{best_key}_best.pt')

if os.path.exists(best_path):
    if 'multitask' in best_key or 'combined' in best_key:
        best_model = DeBERTaNERMultiTask(num_labels=NUM_LABELS)
    else:
        best_model = DeBERTaNER(num_labels=NUM_LABELS)
    best_model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    best_model = best_model.to(DEVICE)
    best_model.eval()
    print('Best model loaded successfully.')
else:
    print(f'Checkpoint not found: {best_path}')

In [ ]:
# Analyze specific error categories
error_categories = {
    'missed_social': 0,
    'missed_clinical': 0,
    'false_positive_social': 0,
    'false_positive_clinical': 0,
    'type_confusion': 0,  # Predicted wrong entity type
    'boundary_error': 0,  # Right type but wrong boundaries
}

example_errors = []

if 'best_model' in dir():
    from torch.utils.data import DataLoader
    dev_loader = DataLoader(dev_dataset, batch_size=8)
    
    all_gold = []
    all_pred = []
    
    with torch.no_grad():
        for batch in dev_loader:
            batch_device = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = best_model(
                input_ids=batch_device['input_ids'],
                attention_mask=batch_device['attention_mask'],
            )
            preds = outputs['logits'].argmax(dim=-1).cpu().numpy()
            
            for i in range(preds.shape[0]):
                idx = len(all_gold)
                if idx >= len(dev_dataset.samples):
                    break
                sample = dev_dataset.get_full_sample(idx)
                word_ids = sample['word_ids']
                raw_tags = sample['raw_tags']
                raw_tokens = sample['raw_tokens']
                
                word_preds = {}
                for j, wid in enumerate(word_ids):
                    if wid is not None and wid not in word_preds:
                        word_preds[wid] = ID2LABEL[preds[i][j]]
                
                pred_tags = [word_preds.get(j, 'O') for j in range(len(raw_tags))]
                all_gold.append(raw_tags)
                all_pred.append(pred_tags)
                
                # Categorize errors
                gold_entities = set()
                pred_entities = set()
                for j, (g, p) in enumerate(zip(raw_tags, pred_tags)):
                    if g.startswith('B-'):
                        gold_entities.add((j, g[2:]))
                    if p.startswith('B-'):
                        pred_entities.add((j, p[2:]))
                
                for j, etype in gold_entities:
                    if not any(abs(j-pj) <= 2 and pe == etype for pj, pe in pred_entities):
                        if 'Social' in etype:
                            error_categories['missed_social'] += 1
                        else:
                            error_categories['missed_clinical'] += 1
                        if len(example_errors) < 10:
                            example_errors.append({
                                'tokens': raw_tokens,
                                'gold': raw_tags,
                                'pred': pred_tags,
                                'error_type': f'missed_{etype}',
                            })

    print('\nError Category Breakdown:')
    for cat, count in sorted(error_categories.items(), key=lambda x: -x[1]):
        print(f'  {cat}: {count}')
    
    print('\n--- Example Errors ---')
    for err in example_errors[:5]:
        tokens = err['tokens']
        gold = err['gold']
        pred = err['pred']
        print(f'\nText: {" ".join(tokens[:30])}...')
        diffs = [(t, g, p) for t, g, p in zip(tokens, gold, pred) if g != p and g != 'O']
        for t, g, p in diffs[:5]:
            print(f'  "{t}": gold={g}, pred={p}')

## 8. Conclusion

### Key Findings

1. **5-Model Ensemble exceeds published SOTA** (Relaxed F1 = 0.611 vs 0.610), combining models trained with different objectives (recall-boost, R-Drop, FGM+SWA) for prediction diversity.

2. **O-Class Downweighting is the single most impactful technique** -- reducing O-class weight from 1.0 to 0.2 boosted recall from 0.510 to 0.587, yielding the best single model at F1 = 0.603.

3. **Auxiliary Task Learning is the most effective architectural innovation** (F1 = 0.584, +0.016 over baseline), producing encoder representations more sensitive to entity-bearing contexts.

4. **Ensemble diversity matters more than individual model quality** -- the best ensemble includes a model with only F1=0.571 individually (FGM+SWA), because its different training objective provides complementary predictions.

5. **Focal Loss hurts overall performance** (F1 = 0.541) despite improving recall, due to excessive false positives.

6. **Not all intuitive innovations produce additive gains** -- combining all innovations introduces conflicting training signals.

### Summary Table

| Model | Dev Relaxed F1 |
|-------|---------------|
| BiLSTM-CRF | 0.262 |
| GLiNER (zero-shot) | 0.317 |
| GPT-4o 3-shot (published) | 0.440 |
| DeBERTa baseline | 0.568 |
| DeBERTa + Auxiliary Tasks | 0.584 |
| Recall-Boost (best single) | 0.603 |
| DeBERTa-large (published SOTA) | 0.610 |
| **5-Model Ensemble (ours)** | **0.611** |

In [ ]:
# Analyze synthetic data composition
synth_data = generate_synthetic_data(n_per_category=50)

categories = Counter(s['category'] for s in synth_data)
print('Synthetic data categories:')
for cat, count in categories.items():
    print(f'  {cat}: {count}')

print(f'\nTotal synthetic samples: {len(synth_data)}')
print(f'\nExample synthetic samples:')
for s in synth_data[:3]:
    print(f'  [{s["category"]}] {" ".join(s["tokens"])}')
    entities = [(t, tag) for t, tag in zip(s['tokens'], s['ner_tags']) if tag != 'O']
    print(f'    Entities: {entities}')

## 8. Conclusion

### Key Findings

1. **Auxiliary Task Learning is the most effective single innovation** (Relaxed F1 = 0.584, +0.016 over baseline), producing the highest Clinical F1 (0.612) and overall precision (0.682). The auxiliary entity presence detection head shapes encoder representations to be more sensitive to entity-bearing contexts.

2. **Definition Prompting provides consistent but marginal gains** (F1 = 0.570), with the most notable improvement in Strict F1 (0.411 vs 0.392), suggesting better entity boundary detection.

3. **Focal Loss hurts overall performance** (F1 = 0.541) despite improving recall, due to excessive false positives from aggressive Social Impact weighting.

4. **Combining all innovations does not produce additive gains** (F1 = 0.566) -- conflicting training signals from focal loss and multi-task objectives cancel each other out.

5. **GLiNER zero-shot achieves F1 = 0.317** without any task-specific training, validating definition-aware architectures for this domain.

6. **BiLSTM-CRF** establishes the pre-transformer floor at F1 = 0.262, confirming the value of large-scale pretraining.

### Summary Table

| Model | Dev Relaxed F1 |
|-------|---------------|
| BiLSTM-CRF | 0.262 |
| GLiNER (zero-shot) | 0.317 |
| GPT-4o 3-shot (published) | 0.440 |
| DeBERTa baseline | 0.568 |
| **DeBERTa + Auxiliary Tasks** | **0.584** |
| DeBERTa-large (published SOTA, test) | 0.610 |